# Clase 8 · ¿Tiene algo que ver una cosa con la otra?

**Estadística Descriptiva e Inferencial** · Módulo 3 · Sesión 8 de 14

Variables categóricas, tablas de contingencia y la prueba χ² («ji cuadrado»).

---

## La pregunta de hoy

De **200 solicitudes de crédito**, 60 se rechazaron. Unas entraron por agencia y otras
por la app.

> **¿El canal por el que entra la solicitud tiene algo que ver con que se rechace,
> o es pura casualidad?**

Esa es toda la clase. Una tabla, una pregunta.

## Lo que cambia respecto a las clases anteriores

Hasta ahora comparábamos **números**: montos, tiempos, gastos. Hoy comparamos
**etiquetas**: aprobado o rechazado, agencia o app.

No se puede calcular el promedio de «agencia». Así que necesitamos otra herramienta.

| | Antes (Clases 4–6) | Hoy |
|---|---|---|
| Los datos son | números | categorías |
| Resumíamos con | media, desviación | **conteos** |
| Comparábamos con | t de Welch, ANOVA | **χ²** |

## El laboratorio

| Bloque | Min | Qué haces |
|---|---|---|
| 1 | 10 | Reproduces con Python el χ² que calculaste a mano |
| 2 | 12 | Una base de 400 solicitudes: construyes la tabla |
| 3 | 12 | **Multiplicas la tabla ×100** y descubres algo incómodo |
| 4 | 11 | Tres canales en vez de dos |

> Todo el código de hoy es **explícito y con bucles visibles**. Nada de trucos: la idea
> es que veas cada número aparecer.

---
## Celda 0 · Preparación

Ejecuta esta celda. No hay que descargar nada: los datos se crean aquí mismo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (8, 4), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    """Compara tu resultado con el esperado."""
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

print("Listo. Empezamos.")

---
# Bloque 1 · Tu tabla, en Python  ·  10 min

Esta es la tabla que construimos en la pizarra:

| | Rechazada | Aprobada | **Total** |
|---|---|---|---|
| **Agencia** | 40 | 60 | **100** |
| **App** | 20 | 80 | **100** |
| **Total** | **60** | **140** | **200** |

Se rechaza el **40 %** de lo que entra por agencia y el **20 %** de lo que entra por app.
Parece que sí hay relación. Vamos a comprobarlo.

### Ejercicio 1.1 — Escribe la tabla y saca los totales

Una tabla de contingencia en Python es simplemente una matriz de numpy.

**Pista:** `tabla.sum(axis=1)` suma cada **fila**. `tabla.sum(axis=0)` suma cada
**columna**. `tabla.sum()` da el total general.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
observado = np.array([[40, 60],
                      [20, 80]])

total_filas    = observado.sum(axis=1)
total_columnas = observado.sum(axis=0)
n              = observado.sum()

print("Totales por fila (agencia, app):", total_filas)
print("Totales por columna (rechazada, aprobada):", total_columnas)
print("Total general:", n)
print()
print("Porcentaje de rechazo en cada canal:")
print(f"  agencia: {40}/{100} = {100*40/100:.0f} %")
print(f"  app    : {20}/{100} = {100*20/100:.0f} %")
print()
print("Y en el total, sin separar por canal:")
print(f"  general: {60}/{200} = {100*60/200:.0f} %")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("total general", n, 200),
     check("total de agencia", total_filas[0], 100),
     check("total de rechazadas", total_columnas[0], 60)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — La tabla ESPERADA

Aquí está la idea clave de toda la clase.

**Pregunta:** ¿cómo se vería la tabla **si el canal no tuviera nada que ver** con el
rechazo?

Si el canal no importara, el 30 % general de rechazo se aplicaría igual a los dos canales.
Como cada canal tiene 100 solicitudes, esperaríamos 30 rechazos en cada uno.

La fórmula que generaliza eso es:

$$\text{esperado} = \frac{\text{total de la fila} \times \text{total de la columna}}{n}$$

Calcúlala con dos bucles, para verla funcionar celda por celda.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
esperado = np.zeros((2, 2))

nombres_fila = ["agencia", "app"]
nombres_col  = ["rechazada", "aprobada"]

for i in range(2):
    for j in range(2):
        esperado[i, j] = total_filas[i] * total_columnas[j] / n
        print(f"  {nombres_fila[i]:8} x {nombres_col[j]:10}: "
              f"{total_filas[i]} x {total_columnas[j]} / {n} = {esperado[i, j]:.0f}")

print()
print("Tabla ESPERADA (si el canal no importara):")
print(pd.DataFrame(esperado, index=nombres_fila, columns=nombres_col))
print()
print("Compara con la tabla OBSERVADA (la real):")
print(pd.DataFrame(observado, index=nombres_fila, columns=nombres_col))
print()
print("Fijate: en el mundo 'sin relacion' los dos canales rechazarian 30 solicitudes.")
print("En la realidad, agencia rechazo 40 y app rechazo 20.")
print("La pregunta es si esa diferencia de 10 es mucho o poco.")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("esperado agencia-rechazada", esperado[0, 0], 30),
     check("esperado agencia-aprobada", esperado[0, 1], 70),
     check("esperado app-rechazada", esperado[1, 0], 30),
     check_bool("los totales de la tabla esperada son iguales a los observados",
                abs(esperado.sum() - observado.sum()) < 0.01,
                "la tabla esperada reparte los MISMOS 200 casos, solo que de otra forma")]
print()
print("1.2 OK" if all(r) else "Revisa 1.2")

### Ejercicio 1.3 — El χ², celda por celda

Ahora medimos **cuánto se aleja** lo observado de lo esperado. Para cada celda:

$$\frac{(\text{observado} - \text{esperado})^2}{\text{esperado}}$$

Y se suman las cuatro.

**¿Por qué al cuadrado?** Para que las diferencias negativas no se cancelen con las
positivas.
**¿Por qué se divide entre lo esperado?** Porque una diferencia de 10 sobre 30 esperados
es mucho; sobre 3 000 esperados no es nada.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
chi2_mano = 0

print("Aporte de cada celda:")
for i in range(2):
    for j in range(2):
        o = observado[i, j]
        e = esperado[i, j]
        aporte = (o - e) ** 2 / e
        chi2_mano = chi2_mano + aporte
        print(f"  {nombres_fila[i]:8} x {nombres_col[j]:10}: "
              f"({o} - {e:.0f})^2 / {e:.0f} = {aporte:.4f}")

print()
print(f"chi2 = suma de los cuatro aportes = {chi2_mano:.4f}")
print()
print("Fijate en cuales celdas aportan mas: las de RECHAZADA.")
print("Tiene sentido, porque es ahi donde la diferencia con lo esperado es relativa-")
print("mente mas grande (10 sobre 30 esperados, contra 10 sobre 70).")

In [ ]:
# ── VERIFICACIÓN 1.3 ─────────────────────────────────────────────────────
r = [check("chi cuadrado calculado a mano", chi2_mano, 9.5238)]
print()
print("Este es el mismo numero que sacaste en la pizarra.")
print("Ahora vamos a ver que scipy da exactamente lo mismo.")
print()
print("1.3 OK" if all(r) else "Revisa 1.3")

### Ejercicio 1.4 — Lo mismo con `scipy`, y una trampa

`scipy.stats.chi2_contingency(tabla)` hace todo lo anterior de un golpe.

> ⚠️ **Atención a esto, que confunde a mucha gente:** en tablas de 2×2, `scipy` aplica
> por defecto una cosa llamada **corrección de Yates**, que cambia un poco el resultado.
> Para que coincida con tu cálculo a mano hay que escribir `correction=False`.
>
> No es que una versión esté mal: son dos convenciones. Pero si no lo sabes, ves dos
> números distintos y no entiendes por qué.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
resultado = stats.chi2_contingency(observado, correction=False)

chi2_scipy = resultado.statistic
p_valor    = resultado.pvalue
grados_lib = resultado.dof

print(f"chi2 = {chi2_scipy:.4f}")
print(f"p    = {p_valor:.6f}")
print(f"gl   = {grados_lib}")
print()
print(f"Tu calculo a mano dio {chi2_mano:.4f}. Coinciden exactamente.")
print()
print("La tabla esperada que scipy calcula por dentro:")
print(pd.DataFrame(resultado.expected_freq, index=nombres_fila, columns=nombres_col))
print()
print("Y ahora mira lo que pasa con la correccion de Yates activada (el DEFAULT):")
con_yates = stats.chi2_contingency(observado)
print(f"  chi2 con Yates = {con_yates.statistic:.4f}  (en vez de {chi2_scipy:.4f})")
print("  Por eso hay que escribir correction=False si quieres reproducir el calculo manual.")
print()
print("DECISION: p = 0.002 es menor que 0.05, asi que rechazamos la hipotesis de que")
print("el canal y el resultado sean independientes. Hay relacion.")

In [ ]:
# ── VERIFICACIÓN 1.4 ─────────────────────────────────────────────────────
r = [check("chi2 de scipy", chi2_scipy, 9.5238),
     check("p-valor", p_valor, 0.002028, tol=1e-5),
     check("grados de libertad", grados_lib, 1),
     check_bool("scipy coincide con tu cálculo a mano",
                abs(chi2_scipy - chi2_mano) < 0.001,
                "si no coinciden, te falto poner correction=False")]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.4")

---
# Bloque 2 · Con datos de verdad  ·  12 min

Hasta ahora te dimos la tabla ya hecha. En el trabajo real vas a tener **una fila por
solicitud**, y la tabla la construyes tú.

Vamos a crear una base de 400 solicitudes con tres columnas: canal, resultado y
segmento.

In [ ]:
# ── DEMOSTRACIÓN: creamos la base ────────────────────────────────────────
g = np.random.default_rng(SEED)
N = 400

canal = g.choice(["agencia", "app", "web"], size=N, p=[0.35, 0.40, 0.25])

# La probabilidad de rechazo depende del canal (por construccion)
prob_rechazo = {"agencia": 0.40, "app": 0.20, "web": 0.28}
resultado_sol = []
for c in canal:
    if g.random() < prob_rechazo[c]:
        resultado_sol.append("rechazada")
    else:
        resultado_sol.append("aprobada")

segmento = g.choice(["nuevo", "recurrente"], size=N, p=[0.45, 0.55])

df = pd.DataFrame({"canal": canal,
                   "resultado": resultado_sol,
                   "segmento": segmento})

print(f"{len(df)} solicitudes")
print()
print(df.head(8).to_string(index=False))

### Ejercicio 2.1 — De la base a la tabla

`pd.crosstab(df["columna_filas"], df["columna_columnas"])` construye la tabla de
contingencia contando las combinaciones.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
tabla = pd.crosstab(df["canal"], df["resultado"])
print("Tabla de contingencia (conteos):")
print(tabla)
print()

# Con totales incluidos, que es como se lee mejor
print("Con totales:")
print(pd.crosstab(df["canal"], df["resultado"], margins=True, margins_name="TOTAL"))
print()

# Y en porcentajes POR FILA, que es lo que responde la pregunta
print("Porcentaje de rechazo dentro de cada canal:")
pct = pd.crosstab(df["canal"], df["resultado"], normalize="index") * 100
print(pct.round(1))
print()
print("OJO con esto: normalize='index' calcula porcentajes por FILA.")
print("Es lo correcto aqui, porque la pregunta es '¿que % de CADA CANAL se rechaza?'.")
print("Si usaras normalize='columns' responderias otra pregunta distinta:")
print("'¿que % de los rechazos vino de cada canal?'. No es lo mismo.")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("total de solicitudes en la tabla", tabla.values.sum(), 400),
     check_bool("la tabla tiene 3 filas (los tres canales)", tabla.shape[0] == 3),
     check_bool("y 2 columnas (aprobada / rechazada)", tabla.shape[1] == 2)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### Ejercicio 2.2 — Aplica el χ² a esta tabla

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
res2 = stats.chi2_contingency(tabla)

print(f"chi2 = {res2.statistic:.4f}")
print(f"p    = {res2.pvalue:.6f}")
print(f"gl   = {res2.dof}")
print()
print("Grados de libertad = (filas - 1) x (columnas - 1) = (3-1) x (2-1) = 2")
print()
print("Tabla esperada:")
print(pd.DataFrame(res2.expected_freq,
                   index=tabla.index, columns=tabla.columns).round(1))
print()
if res2.pvalue < 0.05:
    print("p < 0.05: hay evidencia de relacion entre canal y resultado.")
else:
    print("p >= 0.05: no hay evidencia suficiente de relacion.")
print()
print("Nota: aqui NO hace falta correction=False, porque la correccion de Yates")
print("solo se aplica en tablas de 2x2. Esta es 3x2.")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("grados de libertad", res2.dof, 2),
     check_bool("el chi2 es positivo", res2.statistic > 0),
     check_bool("todas las frecuencias esperadas son mayores que 5",
                (res2.expected_freq > 5).all(),
                "si alguna fuera menor que 5, el chi2 no seria confiable")]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.2")

---
# Bloque 3 · El experimento incómodo  ·  12 min

**Este es el bloque más importante del día.**

Vamos a hacer algo muy simple: tomar tu tabla original y **multiplicar cada celda por
100**. Es decir, imaginar que en lugar de 200 solicitudes tuviéramos 20 000, con
exactamente los mismos porcentajes.

| | Rechazada | Aprobada | | | Rechazada | Aprobada |
|---|---|---|---|---|---|---|
| **Agencia** | 40 | 60 | → | **Agencia** | 4 000 | 6 000 |
| **App** | 20 | 80 | → | **App** | 2 000 | 8 000 |

Los porcentajes **no cambian**: agencia sigue rechazando el 40 % y app el 20 %.

**Pregunta antes de correr nada:** ¿qué le pasará al χ²?

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
observado_x100 = observado * 100
res_x100 = stats.chi2_contingency(observado_x100, correction=False)

print("Tabla original (n = 200):")
print(observado)
print(f"  chi2 = {chi2_scipy:8.4f}   p = {p_valor:.2e}")
print()
print("Tabla x100 (n = 20 000), MISMOS porcentajes:")
print(observado_x100)
print(f"  chi2 = {res_x100.statistic:8.4f}   p = {res_x100.pvalue:.2e}")
print()
print("=" * 64)
print(f"El chi2 se multiplico por {res_x100.statistic/chi2_scipy:.0f}.")
print(f"El p-valor paso de {p_valor:.4f} a {res_x100.pvalue:.1e}.")
print("Y los porcentajes de rechazo siguen siendo EXACTAMENTE 40 % y 20 %.")
print("=" * 64)
print()
print("Conclusion incomoda: el chi2 depende del TAMANO de la muestra tanto como")
print("de la fuerza de la relacion. Con muchos datos, cualquier diferencia por")
print("pequena que sea sale 'significativa'.")
print()
print("Esto ya lo vimos en la Clase 5 con el p-valor. Aqui vuelve a aparecer.")

In [ ]:
# ── VERIFICACIÓN 3 ──────────────────────────────────────────────────────
r = [check("chi2 de la tabla x100", res_x100.statistic, 952.38, tol=0.1),
     check_bool("el chi2 se multiplicó por 100",
                abs(res_x100.statistic / chi2_scipy - 100) < 0.5),
     check_bool("el p-valor se volvió diminuto", res_x100.pvalue < 1e-100)]
print()
print("Y ahora la pregunta que abre el bloque 4:")
print("  si el chi2 no me dice cuan fuerte es la relacion... ¿que me lo dice?")
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa el bloque 3")

---
# Bloque 4 · Cramér's V: cuánta relación hay  ·  11 min

El χ² responde **«¿hay relación?»**. La V de Cramér responde **«¿de qué tamaño?»**.

$$V = \sqrt{\frac{\chi^2}{n \times \min(\text{filas}-1,\ \text{columnas}-1)}}$$

Va siempre de **0 a 1**:

| V | Interpretación |
|---|---|
| ~0.1 | relación débil |
| ~0.3 | relación moderada |
| ~0.5 o más | relación fuerte |

Fíjate en que **divide entre n**. Eso es justo lo que le faltaba al χ².

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def cramers_v(tabla_np, chi2):
    """Calcula la V de Cramér a partir de la tabla y su chi2."""
    n_total = tabla_np.sum()
    filas, columnas = tabla_np.shape
    k = min(filas - 1, columnas - 1)
    return np.sqrt(chi2 / (n_total * k))

v_original = cramers_v(observado, chi2_scipy)
v_x100     = cramers_v(observado_x100, res_x100.statistic)

print(f"{'':22}{'n':>8}{'chi2':>12}{'p':>14}{'V':>10}")
print("-" * 66)
print(f"{'tabla original':22}{200:>8}{chi2_scipy:>12.2f}{p_valor:>14.2e}{v_original:>10.4f}")
print(f"{'tabla x100':22}{20000:>8}{res_x100.statistic:>12.2f}{res_x100.pvalue:>14.2e}{v_x100:>10.4f}")
print("-" * 66)
print()
print("MIRA LA ULTIMA COLUMNA.")
print(f"El chi2 se multiplico por 100 y el p se desplomo, pero V no se movio:")
print(f"{v_original:.4f} contra {v_x100:.4f}. Identicos.")
print()
print("Eso es porque V mide la FUERZA de la relacion, y la fuerza no cambio:")
print("agencia sigue rechazando el doble que app en las dos tablas.")
print()
print(f"Interpretacion: V = {v_original:.2f} es una relacion DEBIL-MODERADA.")
print("El canal importa, pero explica solo una parte de por que se rechaza")
print("una solicitud. Hay mas factores en juego que no estamos midiendo.")

In [ ]:
# ── VERIFICACIÓN 4 ──────────────────────────────────────────────────────
r = [check("V de la tabla original", v_original, 0.2182),
     check("V de la tabla x100", v_x100, 0.2182),
     check_bool("las dos V son iguales", abs(v_original - v_x100) < 0.0001,
                "si no salen iguales, revisa que dividas entre el n de CADA tabla"),
     check_bool("V está entre 0 y 1", 0 <= v_original <= 1)]
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa el bloque 4")

### Ejercicio 4.2 — Aplícalo a la base de 400

Calcula la V para la tabla de tres canales del bloque 2, y también para una relación
que **no** debería existir: canal contra segmento.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
# a) canal vs resultado
v_canal_resultado = cramers_v(tabla.values, res2.statistic)

# b) canal vs segmento
tabla_seg = pd.crosstab(df["canal"], df["segmento"])
res_seg   = stats.chi2_contingency(tabla_seg)
v_canal_segmento = cramers_v(tabla_seg.values, res_seg.statistic)

print("a) CANAL vs RESULTADO   (esperamos que SI haya relacion)")
print(f"   chi2 = {res2.statistic:.2f}   p = {res2.pvalue:.4f}   V = {v_canal_resultado:.4f}")
print()
print("b) CANAL vs SEGMENTO    (los generamos independientes: NO deberia haber)")
print(tabla_seg)
print(f"   chi2 = {res_seg.statistic:.2f}   p = {res_seg.pvalue:.4f}   V = {v_canal_segmento:.4f}")
print()
if res_seg.pvalue >= 0.05:
    print("Como esperabamos: p >= 0.05, no hay evidencia de relacion, y V es chiquita.")
else:
    print("Curioso: salio significativo aunque los generamos independientes.")
    print("Es un FALSO POSITIVO, y ocurre el 5 % de las veces. Clase 5.")
print()
print("Compara las dos V: la relacion real es varias veces mas fuerte que el ruido.")

In [ ]:
# ── VERIFICACIÓN 4.2 ────────────────────────────────────────────────────
r = [check_bool("V de canal-resultado es mayor que V de canal-segmento",
                v_canal_resultado > v_canal_segmento),
     check_bool("V de canal-segmento es pequeña (< 0.15)",
                v_canal_segmento < 0.15,
                "los generamos independientes, así que su V debe ser cercana a 0")]
print()
print("LABORATORIO COMPLETO" if all(r) else "Revisa 4.2")

---
# Cierre

### Los cuatro pasos del χ², en orden

1. **Tabla observada** — cuenta las combinaciones (`pd.crosstab`)
2. **Tabla esperada** — `fila × columna / n` en cada celda
3. **χ²** — suma de `(observado − esperado)² / esperado`
4. **Decisión** — `p < 0.05` → hay relación. Y después, **siempre**, calcula V

### Checklist de salida

- [ ] Sé construir una tabla de contingencia desde una base de datos.
- [ ] Entiendo qué significa la tabla «esperada».
- [ ] Sé calcular el χ² a mano y con `scipy`.
- [ ] Sé que en 2×2 hay que poner `correction=False` para reproducir el cálculo manual.
- [ ] Sé que el χ² depende del tamaño de muestra tanto como de la fuerza de la relación.
- [ ] Reporto siempre la V de Cramér junto al p-valor.

### Lo que quedó demostrado

| | |
|---|---|
| χ² a mano y con scipy | **9.5238**, idénticos |
| Tabla ×100: χ² | de 9.52 a **952.38** (×100) |
| Tabla ×100: V | **0.2182 → 0.2182**, sin moverse |

Los porcentajes de rechazo eran los mismos en las dos tablas. **Lo único que cambió fue
n**, y eso bastó para que el p-valor pasara de 0.002 a 4 × 10⁻²⁰⁹.

### Errores frecuentes

| Error | Por qué está mal |
|---|---|
| «p < 0.001, la relación es fuertísima» | El p mide evidencia, no fuerza. Para fuerza, V |
| Usar porcentajes en `chi2_contingency` | La prueba necesita **conteos**, no porcentajes |
| Ignorar que alguna esperada es < 5 | El χ² deja de ser confiable. Se usa Fisher exacto |
| Concluir causalidad | «Hay relación» no es «el canal causa el rechazo» |

### Reto para la próxima clase

Busca en tu trabajo dos variables categóricas que sospeches relacionadas: canal y
resultado, segmento y mora, región y tipo de producto.

Construye la tabla, calcula el χ² y la V, y contesta: **¿la relación es significativa?
¿y es fuerte?** No siempre las dos respuestas coinciden.

### Clase 9

Nos quedamos con tres cosas pendientes que hoy solo nombramos: **Fisher exacto** para
cuando hay celdas con pocos casos, los **residuos** para saber *qué celda* causa la
diferencia, y la **paradoja de Simpson**, donde una relación se invierte al separar por
grupos.

---
*Estadística Descriptiva e Inferencial · Módulo 3 · Clase 8*